# The Agent Loop and Its Observed Behaviour

**Technical report 2 of the switchgear-estimation-assistant project**

| Item | Detail |
|---|---|
| Status | In progress |
| Covers | `run_agent` in `agent.py` and `schemas.py`, both in `ai_engine/estimation_assistant/` |
| Builds on | Report 1: tool layer and safe execution |
| Data | Synthetic only |

## 1. Purpose

Report 1 described the tools and the runner that executes them. This report describes the loop that connects them to a language model: it asks the model, runs the tools the model requests, returns the results, and repeats until the model answers in text or a round limit is reached.

## 2. The loop

```
messages = [question]
repeat up to max_rounds times:
    ask the model with the full history and the tool list
    save the model's reply in the history
    if the reply has no tool request: return its text
    for each tool request:
        run it with run_tool
        save the result in the history, with the role "tool"
stop with a message if the limit is reached
```

| Element | Purpose |
|---|---|
| Full history on every call | The model keeps no memory between calls, so every call carries the whole conversation |
| Reply saved before the stop check | A tool request is stored as well as a text answer, so a result always follows its request |
| `run_tool` for every request | An unknown tool or wrong inputs return a message, not an exception |
| `max_rounds` | A run cannot continue indefinitely |

## 3. Implementation

```python
def run_agent(question: str, max_rounds: int = 5) -> AgentResult:
    """Ask the model, run the tools it requests, and return the answer with the steps taken."""
    messages = [{"role": "user", "content": question}]
    steps: list[ToolStep] = []

    for round_number in range(1, max_rounds + 1):
        print(f"Round {round_number}")

        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=list(TOOLS.values()),
            think=False,
        )
        messages.append(response.message)

        if not response.message.tool_calls:
            return build_result(question, response.message.content or "", steps)

        for call in response.message.tool_calls:
            name = call.function.name
            arguments = dict(call.function.arguments)
            result = run_tool(name, arguments)
            print(f"  {name}({arguments}) -> {result}")
            steps.append(ToolStep(name=name, arguments=arguments, result=result))
            messages.append({"role": "tool", "tool_name": name, "content": result})

    return build_result(question, "Stopped: reached the round limit.", steps)
```

The result is typed. `schemas.py` defines the shape.

```python
class ToolStep(BaseModel):
    name: str
    arguments: dict
    result: str

class AgentResult(BaseModel):
    answer: str
    steps: list[ToolStep]
    sources: list[str] = []
    unsupported: list[float] = []
```

| Field | Content |
|---|---|
| `answer` | The model's final text |
| `steps` | One `ToolStep` per tool run, in order, each with name, inputs and result |
| `sources` | The note ids returned by `search_notes`, filled by code from the steps |
| `unsupported` | Numbers in the answer that appear nowhere in the question or the steps |

Because every step is recorded, an answer can be checked against the log afterwards.

Thinking mode is switched off (`think=False`). In an earlier test, a model with a small token budget spent all of it on reasoning and returned an empty answer.

### 3.1 Three details of the agent loop

### `TOOLS.values()`

| Call | Gives |
|---|---|
| `TOOLS.keys()` | The tool **names**, as text. The model sends these back. |
| `TOOLS.values()` | The real **functions**. `ollama.chat(tools=...)` needs these. |
| `list(TOOLS.values())` | The same, as a plain list |

Using the dictionary means a new tool is added in one place only.

### `think`

| Setting | Effect |
|---|---|
| `think=True` | The model writes its reasoning first. Slower. Stored in `response.message.thinking`. |
| `think=False` | The model answers directly. Faster. Used for tool calls and structured output. |

In an earlier test the model spent its whole token budget on thinking and returned an empty answer.

### `messages.append(response.message)`

The model has no memory of its own. The `messages` list is the only memory, and it is sent in full every round.

| Without the append | With the append |
|---|---|
| The tool result appears with no request before it | The history reads: question, request, result, answer |

The append happens **before** the stop check, so every reply is saved, both text answers and tool requests.

### 3.2 The stop rule

```python
if not response.message.tool_calls:
    return response.message.content
```

**Question it answers:** after each model reply, did the model ask for a tool?

| `tool_calls` | `not tool_calls` | Result |
|---|---|---|
| `None` or `[]` | `True` | The `return` runs. The text answer is returned and the loop ends. |
| A list with requests | `False` | The `if` is skipped. The loop goes on to the next round. |

**Why `not` works:** `None` and an empty list count as "empty". `not` of an empty value is `True`.

**Why it is inside the loop:** any round might be the last, so the check must run after every reply.

## 4. Environment

| Item | Detail |
|---|---|
| Model | Gemma 4, small size, served by Ollama |
| Hardware | Google Colab, Tesla T4 |
| Files | The five files in `ai_engine/estimation_assistant/`: `schemas.py`, `tools.py`, `retrieval.py`, `checks.py`, `agent.py` |

## 5. Observed runs

| Question | Output | Reading |
|---|---|---|
| What is the capital of France? | `Round 1`, then a text answer | No tool needed. The stop rule ended the run in one round. |
| What is the price of one busbar? | `Round 1`, `lookup_price({'part_name': 'busbar'}) -> 95.0`, `Round 2`, then a text answer of 95.0 | One tool request, then an answer based on its result |
| How much do 3 busbars cost in total? | `Round 1`, `lookup_price(...) -> 95.0`, `Round 2`, `line_total({'quantity': 3, 'unit_price': 95}) -> 285`, `Round 3`, then a text answer of 285.00 | Two tools in sequence. The first result became the second tool's input. |
| Call the `list_parts` tool and show what it returns | `Round 1`, `list_parts({}) -> mccb, acb, ...`, `Round 2`, then an **empty** answer | The tool ran, but the model returned no text. The structured result still holds the step. |
| How much do 3 busbars cost in total? (typed result) | `result.answer` is the text. `result.steps` holds the two `ToolStep` records. | The log can be read without parsing text |

**Direct checks of `run_tool`**

| Request | Result |
|---|---|
| Unknown tool `make_coffee` | Message listing the three valid tools |
| `lookup_price` for `fuse` | Message listing the five valid parts |
| `line_total` without `quantity` | Message containing Python's error text |

**Check applied to every answer:** each number in the final text must match a logged tool result. `95.0` and `285` do.

## 6. Findings

1. **A stale module copy hid the loop.** An earlier run showed two rounds and no tool lines. The cause was an outdated `agent.py` loaded in the session, not the model. Writing the new file and reloading the module resolved it. Checks that distinguish the cases: a text search of the file, and `inspect.getsource` on the loaded function.
2. **Tool descriptions changed behaviour.** In an earlier prototype, a short description without the valid part names made the model ask a clarifying question. Listing the names in the docstring led to a tool call.
3. **Formatting is chosen by the model.** The tool returned `285`. The final text said `285.00`. Presentation of amounts should be done by code before it reaches the user.
4. **Types are not guaranteed.** `lookup_price` returns text. The model passed `95` as a number to `line_total`. This worked in the observed run but is not enforced.
5. **Descriptions can replace tools.** Part names listed in a tool description let the model answer a catalogue question without calling the listing tool. The answer is only as current as the description, so the description and the data must be kept in step.
6. **A tool with no inputs works.** `list_parts` was accepted and run when requested explicitly. Without that request, the model answered from the description and did not call it.

## 7. Limitations

| Limitation | Note |
|---|---|
| Few runs | Three questions on one model. This shows the loop works, not how often. |
| Small model | It may skip a tool and answer from its own knowledge. The log check catches this. |
| Number check is limited | `checks.py` compares the digits in `answer` with `steps`. It does not prove a number is used correctly. See the Limits section of the README. |
| Empty replies | After a tool result the model may return no text. The caller should handle an empty `answer`. |
| One price per part type | Real quotes depend on ratings and sizes |
| Synthetic data | Results say nothing about real pricing |

## 8. Next steps

| Step | Content | Status |
|---|---|---|
| 1 | Output checks: every number in the answer must match a logged tool result | Done (`checks.py`) |
| 2 | Retrieval over a small synthetic catalogue, with the source kept in the run result | Done (report 3) |
| 3 | Reading drawing images | Not started |